# Attention Head Lineage

Trace how attention heads compose across layers: head-to-head attention,
output influence on downstream Q/K/V, composition chains, and dependency graphs.

## Why This Matters

Attention head lineage reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_lineage import (
    head_to_head_attention, head_output_influence,
    head_composition_chain, layer_head_dependency_graph,
    attention_head_lineage_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Head-to-Head Attention

How much does each head in layer 1 read from each head in layer 0?

In [ ]:
result = head_to_head_attention(model, tokens, src_layer=0, dst_layer=1)
print('Strongest connections:')
for c in result['strongest_connections'][:6]:
    bar = '█' * int(c['strength'] * 200)
    print(f"  L0H{c['src_head']} -> L1H{c['dst_head']}: {c['strength']:.4f} {bar}")

## Head Output Influence

Does each head's output project more onto downstream Q, K, or V?

In [ ]:
result = head_output_influence(model, tokens, layer=0)
for h in result['per_head']:
    print(f"  Head {h['head']}: Q={h['q_influence']:.4f} K={h['k_influence']:.4f} "
          f"V={h['v_influence']:.4f} -> {h['dominant_path']}")

## Composition Chains

Trace the strongest head-to-head path through the model.

In [ ]:
for start_head in range(4):
    result = head_composition_chain(model, tokens, start_layer=0, start_head=start_head)
    path = ' -> '.join(f"L{s['layer']}H{s['head']}" for s in result['chain'])
    print(f"  {path}  (strength={result['chain_strength']:.6f})")

## Full Dependency Graph

In [ ]:
result = layer_head_dependency_graph(model, tokens)
print(f"Total edges: {len(result['edges'])}")
for edge in result['edges'][:10]:
    bar = '█' * int(edge['strength'] * 200)
    print(f"  L{edge['src_layer']}H{edge['src_head']} -> "
          f"L{edge['dst_layer']}H{edge['dst_head']}: {edge['strength']:.4f} {bar}")

## Lineage Summary

In [ ]:
result = attention_head_lineage_summary(model, tokens)
print(f"Strong edges: {result['n_strong_edges']}")
print(f"Mean edge strength: {result['mean_edge_strength']:.6f}")
for i, chain in enumerate(result['strongest_chains']):
    path = ' -> '.join(f"L{s['layer']}H{s['head']}" for s in chain['chain'])
    print(f"  Chain {i+1}: {path} (str={chain['chain_strength']:.6f})")